# 04 — Model Training & Evaluation

**Prerequisite**: `notebooks/03_clustering.ipynb` must be complete.
The following files must exist:
- `data/processed/zone_hour_grid.parquet`
- `data/processed/zone_day_grid.parquet`
- `data/processed/cis_table.parquet`

**What happens here**:
1. Load aggregated zone×time grids
2. Apply time-based train/test split (train ≤ 2024-02-29 / test ≥ 2024-03-01)
3. Assert no-future-leakage (hard error if violated)
4. Train XGBoost, LightGBM, CatBoost on both hour and day resolutions
5. Evaluate each model: MAE, RMSE, Precision@K, NDCG@10 vs frequency baseline
6. Select winner by NDCG@10 and update configs/model.yaml
7. Save all checkpoints

**Files saved**:
- `checkpoints/{model}_{resolution}_{timestamp}/` — one folder per trained model
- `data/outputs/eval_{timestamp}.json` — full evaluation results
- `configs/model.yaml` — updated with winner

## Cell 1 — Environment setup
**Expected output**: `Project root: ...GridLock R2`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\USER\Desktop\GridLock R2


## Cell 2 — Configure loguru
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='DEBUG',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Verify prerequisite files exist
**Expected output**: All 3 files found with sizes printed.

In [3]:
required_files = [
    PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'zone_day_grid.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet',
]

all_ok = True
for f in required_files:
    if f.exists():
        print(f'  ✓  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)')
    else:
        print(f'  ✗  MISSING: {f.name} — run 03_clustering.ipynb first!')
        all_ok = False

if not all_ok:
    raise FileNotFoundError('One or more prerequisite files are missing. See above.')
print('\nAll prerequisite files found. Proceed.')

  ✓  zone_hour_grid.parquet  (148.2 KB)
  ✓  zone_day_grid.parquet  (54.2 KB)
  ✓  cis_table.parquet  (9.0 KB)

All prerequisite files found. Proceed.


## Cell 4 — Quick data preview
**What this cell does**: Load and preview both grids before training.  
**Expected output**: Shape and column list for hour and day grids.

In [4]:
import pandas as pd

hour_df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet')
day_df  = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'zone_day_grid.parquet')
cis_df  = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet')

print(f'Zone×Hour grid : {hour_df.shape}  |  cols: {list(hour_df.columns)}')
print(f'Zone×Day  grid : {day_df.shape}   |  cols: {list(day_df.columns)}')
print(f'CIS table      : {cis_df.shape}')
print(f'\nHour target stats:')
print(hour_df['zone_hour_violation_count'].describe())
print(f'\nDay target stats:')
print(day_df['zone_day_violation_count'].describe())

Zone×Hour grid : (26354, 12)  |  cols: ['zone_id', 'date', 'hour_of_day', 'zone_hour_violation_count', 'fraction_at_junction', 'dominant_violation_type', 'dominant_vehicle_type', 'police_station_id', 'center_code_encoded', 'data_sent_to_scita_mean', 'is_weekend', 'month']
Zone×Day  grid : (8246, 11)   |  cols: ['zone_id', 'date', 'zone_day_violation_count', 'fraction_at_junction', 'dominant_violation_type', 'dominant_vehicle_type', 'police_station_id', 'center_code_encoded', 'data_sent_to_scita_mean', 'is_weekend', 'month']
CIS table      : (140, 8)

Hour target stats:
count    26354.000000
mean        10.179897
std         24.209280
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max        284.000000
Name: zone_hour_violation_count, dtype: float64

Day target stats:
count    8246.000000
mean       32.534683
std       148.376908
min         1.000000
25%         1.000000
50%         4.000000
75%        15.000000
max      1848.000000
Name: zone_da

## Cell 5 — Run full training pipeline
**What this cell does**: Calls `src/training/train.run_training()` which:
- Applies time-based split (train ≤ 2024-02-29 / test ≥ 2024-03-01)
- Asserts no-future-leakage (hard error if violated)
- Trains XGBoost, LightGBM, CatBoost on BOTH hour and day resolutions (6 runs total)
- Evaluates each run: MAE, RMSE, Precision@5/10, NDCG@5/10 vs baseline
- Saves checkpoints

**Expected runtime**: ~5–10 minutes total (LightGBM fastest, CatBoost slowest).

**Expected output**: Training logs per model, then comparison table, then winner.

In [5]:
from src.training.train import run_training

results = run_training(project_root=PROJECT_ROOT)

print('\n=== Training complete ===')
print(f"Winner: {results['winner']['run']}")
print(f"  NDCG@10     : {results['winner']['NDCG@10']:.4f}")
print(f"  Precision@10: {results['winner']['Precision@10']:.4f}")
print(f"  MAE         : {results['winner']['MAE']:.4f}")

21:24:47 | INFO     | Configs loaded | features.yaml hash: 8529a19f7bf2e3aa...
21:24:47 | INFO     | Training run started | timestamp=20260616_155447 | seed=42
21:24:47 | INFO     | CIS table loaded: 140 zones


Training all models:   0%|          | 0/6 [00:00<?, ?run/s]

21:24:47 | INFO     | 
  Training: XGBOOST | resolution=hour
  Target: zone_hour_violation_count | Features: 10
21:24:48 | INFO     | Split: train=19,870 rows (2023-11-09 → 2024-02-29) | test=6,484 rows (2024-03-01 → 2024-04-08) | Leakage guard: PASSED ✓
21:24:48 | INFO     | X_train: (19870, 10)  y_train mean=10.30 max=284
21:24:48 | INFO     | X_val:   (6484, 10)    y_val mean=9.82 max=266


21:24:49 | INFO     | === Evaluating: xgboost | hour | target='zone_hour_violation_count' ===
21:24:49 | INFO     | [xgboost/hour] MAE=4.6820  RMSE=10.6612
21:24:49 | INFO     | Relevance assigned: grade=2 → 35 zones | grade=1 → 35 zones | grade=0 → 67 zones (q50=36.0, q75=113.0)
21:24:49 | INFO     |   [xgboost] NDCG@5=1.0000  Precision@5=1.0000
21:24:49 | INFO     |   [xgboost] NDCG@10=1.0000  Precision@10=1.0000
21:24:49 | INFO     | Frequency baseline: 140 zones | top-3 zones: {2: 185578.5, 7: 1537.732546, 4: 160.647921}
21:24:49 | INFO     |   [baseline] NDCG@5=1.0000  Precision@5=1.0000
21:24:49 | INFO     |   [baseline] NDCG@10=1.0000  Precision@10=1.0000
21:24:49 | WARNING  | ⚠ xgboost/hour does NOT beat baseline on NDCG. Investigate features.
21:24:50 | INFO     | Checkpoint saved → 'C:\Users\USER\Desktop\GridLock R2\checkpoints\xgboost_hour_20260616_155447'


C:\Users\USER\Desktop\GridLock R2\venv\Lib\site-packages\xgboost\sklearn.py:1116: UserWarning: [21:24:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
Training all models:  17%|█▋        | 1/6 [00:02<00:10,  2.02s/run]

...

Training all models:  83%|████████▎ | 5/6 [00:06<00:01,  1.16s/run]

21:24:54 | INFO     | 
  Training: CATBOOST | resolution=day
  Target: zone_day_violation_count | Features: 9
21:24:54 | INFO     | Split: train=6,181 rows (2023-11-09 → 2024-02-29) | test=2,065 rows (2024-03-01 → 2024-04-08) | Leakage guard: PASSED ✓
21:24:54 | INFO     | X_train: (6181, 9)  y_train mean=33.10 max=1848
21:24:54 | INFO     | X_val:   (2065, 9)    y_val mean=30.83 max=1523


21:24:54 | INFO     | === Evaluating: catboost | day | target='zone_day_violation_count' ===
21:24:54 | INFO     | [catboost/day] MAE=13.3972  RMSE=29.2547
21:24:54 | INFO     | Relevance assigned: grade=2 → 35 zones | grade=1 → 35 zones | grade=0 → 67 zones (q50=36.0, q75=113.0)
21:24:54 | INFO     |   [catboost] NDCG@5=1.0000  Precision@5=1.0000
21:24:54 | INFO     |   [catboost] NDCG@10=1.0000  Precision@10=1.0000
21:24:55 | INFO     | Frequency baseline: 140 zones | top-3 zones: {2: 185578.5, 7: 1537.732546, 4: 160.647921}
21:24:55 | INFO     |   [baseline] NDCG@5=1.0000  Precision@5=1.0000
21:24:55 | INFO     |   [baseline] NDCG@10=1.0000  Precision@10=1.0000
21:24:55 | WARNING  | ⚠ catboost/day does NOT beat baseline on NDCG. Investigate features.
21:24:55 | INFO     | Checkpoint saved → 'C:\Users\USER\Desktop\GridLock R2\checkpoints\catboost_day_20260616_155447'


Training all models: 100%|██████████| 6/6 [00:07<00:00,  1.17s/run]

21:24:55 | INFO     | 
  MODEL COMPARISON SUMMARY
21:24:55 | INFO     |   xgboost_hour                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.6820
21:24:55 | INFO     |   lightgbm_hour                  | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.7238
21:24:55 | INFO     |   catboost_hour                  | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.9967
21:24:55 | INFO     |   xgboost_day                    | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=10.4586
21:24:55 | INFO     |   lightgbm_day                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=10.5859
21:24:55 | INFO     |   catboost_day                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=13.3972
21:24:55 | INFO     | 
  🏆 WINNER: xgboost_hour | NDCG@10=1.0000 | Precision@10=1.0000 | MAE=4.6820
21:24:55 | INFO     | Eval results saved → 'C:\Users\USER\Desktop\GridLock R2\data\outputs\eval_20260616_155447.json'
21:24:55 | INFO     | model.yaml updated: 'primary_model: null' → 'primary_model: "xgboost"'
21:24:55 | INFO     | mo

## Cell 6 — Comparison table
**What this cell does**: Print a formatted table of all 6 model runs ranked by NDCG@10.  
**Expected output**: Table with 6 rows (3 models × 2 resolutions), sorted by NDCG@10.

In [6]:
import pandas as pd

comparison_df = pd.DataFrame(results['comparison_table'])
comparison_df = comparison_df.sort_values('NDCG@10', ascending=False).reset_index(drop=True)

print('=== Model Comparison (sorted by NDCG@10) ===')
print(comparison_df[['run', 'NDCG@10', 'Precision@10', 'MAE', 'RMSE']].to_string(index=False))
print(f"\nWinner → {results['winner']['run']}")

=== Model Comparison (sorted by NDCG@10) ===
          run  NDCG@10  Precision@10       MAE      RMSE
 xgboost_hour      1.0           1.0  4.681994 10.661168
lightgbm_hour      1.0           1.0  4.723807 10.610725
catboost_hour      1.0           1.0  4.996702 11.347834
  xgboost_day      1.0           1.0 10.458567 28.064938
 lightgbm_day      1.0           1.0 10.585875 28.097664
 catboost_day      1.0           1.0 13.397197 29.254742

Winner → xgboost_hour


## Cell 7 — Baseline vs ML comparison
**What this cell does**: For each model, show whether it beats the frequency ranker baseline on NDCG@10 and Precision@10.  
**Expected output**: ✓ or ✗ for each model vs baseline.

In [7]:
print('=== Baseline Beat Report ===')
print(f'{"Run":<35} {"Beats NDCG":>12} {"Beats Prec":>12}')
print('-' * 62)
for row in results['comparison_table']:
    ndcg_ok = '✓' if row['beats_baseline']['ndcg']      else '✗'
    prec_ok = '✓' if row['beats_baseline']['precision'] else '✗'
    print(f"{row['run']:<35} {ndcg_ok:>12} {prec_ok:>12}")

=== Baseline Beat Report ===
Run                                   Beats NDCG   Beats Prec
--------------------------------------------------------------
xgboost_hour                                   ✗            ✗
lightgbm_hour                                  ✗            ✗
catboost_hour                                  ✗            ✗
xgboost_day                                    ✗            ✗
lightgbm_day                                   ✗            ✗
catboost_day                                   ✗            ✗


## Cell 8 — Checkpoint verification
**What this cell does**: Verify all checkpoint folders were created and contain expected files.  
**Expected output**: List of saved checkpoint directories.

In [8]:
from pathlib import Path

ckpt_root = PROJECT_ROOT / 'checkpoints'
print(f'Checkpoint directory: {ckpt_root}')
print()

for folder in sorted(ckpt_root.iterdir()):
    if folder.is_dir():
        files = list(folder.iterdir())
        print(f'  {folder.name}/')
        for f in sorted(files):
            print(f'    {f.name}  ({f.stat().st_size / 1e3:.1f} KB)')

Checkpoint directory: C:\Users\USER\Desktop\GridLock R2\checkpoints

  catboost_day_20260616_155447/
    eval.yaml  (8.9 KB)
    features.yaml  (6.2 KB)
    label_encoders.pkl  (2.2 KB)
    model.cbm  (82.4 KB)
    model.yaml  (8.8 KB)
    training_meta.json  (1.0 KB)
  catboost_hour_20260616_155447/
    eval.yaml  (8.9 KB)
    features.yaml  (6.2 KB)
    label_encoders.pkl  (2.2 KB)
    model.cbm  (349.7 KB)
    model.yaml  (8.8 KB)
    training_meta.json  (1.0 KB)
  lightgbm_day_20260616_155447/
    eval.yaml  (8.9 KB)
    features.yaml  (6.2 KB)
    label_encoders.pkl  (2.2 KB)
    model.lgb  (293.1 KB)
    model.yaml  (8.8 KB)
    training_meta.json  (1.0 KB)
  lightgbm_hour_20260616_155447/
    eval.yaml  (8.9 KB)
    features.yaml  (6.2 KB)
    label_encoders.pkl  (2.2 KB)
    model.lgb  (489.5 KB)
    model.yaml  (8.8 KB)
    training_meta.json  (1.0 KB)
  xgboost_day_20260616_155447/
    eval.yaml  (8.9 KB)
    features.yaml  (6.2 KB)
    label_encoders.pkl  (2.2 KB)
    model.

## Summary

**What was done**:
- Time-based split applied (train ≤ 2024-02-29 / test ≥ 2024-03-01)
- No-future-leakage guard: PASSED
- XGBoost, LightGBM, CatBoost trained on zone×hour and zone×day grids
- Each model evaluated on MAE, RMSE, NDCG@10, Precision@10 vs frequency baseline
- Winner selected by highest NDCG@10 (MAE as tiebreaker)

**Files saved**:
- `checkpoints/{model}_{resolution}_{timestamp}/` (6 checkpoint folders)
- `data/outputs/eval_{timestamp}.json`
- `configs/model.yaml` (updated with winner)

**Next step**: `notebooks/05_inference.ipynb` — load the winning model and generate enforcement priority rankings

In [9]:
print('=== Training Summary ===')
print(f"  Timestamp     : {results['training_timestamp']}")
print(f"  Winner model  : {results['winner']['model']}")
print(f"  Winner resol. : {results['winner']['resolution']}")
print(f"  NDCG@10       : {results['winner']['NDCG@10']:.4f}")
print(f"  Precision@10  : {results['winner']['Precision@10']:.4f}")
print(f"  MAE           : {results['winner']['MAE']:.4f}")
print(f"  RMSE          : {results['winner']['RMSE']:.4f}")
print(f"  Eval output   : data/outputs/eval_{results['training_timestamp']}.json")
print(f"  Next notebook : notebooks/05_inference.ipynb")

=== Training Summary ===
  Timestamp     : 20260616_155447
  Winner model  : xgboost
  Winner resol. : hour
  NDCG@10       : 1.0000
  Precision@10  : 1.0000
  MAE           : 4.6820
  RMSE          : 10.6612
  Eval output   : data/outputs/eval_20260616_155447.json
  Next notebook : notebooks/05_inference.ipynb
